# Talanta walkthrough chapter render (internal, private)

One run renders one frame range of one chapter and one cut (wide 1920 x 1080 or portrait 720 x 1280) in one Blender process.
Order and GPU hours: blender/landing_reel/RENDER_ORDER.md, version C.

How to run:
- Settings: Accelerator GPU T4 x2, Internet on, input the private dataset `talanta-render-pack` (render pack 7e19362).
- Set the variables in the first code cell. Keep each run under 4 h (RENDER_ORDER.md gives the ranges).
- Click "Save Version", then "Save & Run All (Commit)". Keep the version private. The run continues in the background and the frames stay as the version's output.
- Download the output (frames/<run>/), then stop. Do not leave an idle session open.

People, vehicles and lighting in the frames are Visualization. The frames are internal only. Do not publish this notebook or its output.

In [ ]:
# Settings of this run
CHAPTER = "ch1_arrival"   # ch1_arrival (192 frames) or ch2_orbit (240 frames)
CUT = "wide"              # wide: 1920 x 1080; portrait: 720 x 1280, lens shifted to the app's phone L07 window
FRAME_START = 0           # first frame, included
FRAME_END = 23            # last frame, included
SAMPLES = 128             # stills quality
HAZE = 1                  # 1: haze volume and its pass layer (stills quality); 0: compositor haze only
RES = None if CUT == "wide" else "720x1280"     # wide uses the pack default (scale 50 of 3840 x 2160)
SHIFT_X = None if CUT == "wide" else 0.151      # chapters.json "portrait": crop x 820 to 1427 of 1920
RUN = f"{CHAPTER}_{CUT}_{FRAME_START:03d}-{FRAME_END:03d}"
print(RUN)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /tmp /kaggle/working | tail -2; free -g | head -2

In [ ]:
# Blender and the pack go to /tmp, so that the version output holds the frames only.
import glob, json, os, shutil, subprocess, time
WORK = "/tmp/talanta"
BLENDER_URL = "https://download.blender.org/release/Blender5.2/blender-5.2.2-linux-x64.tar.xz"  # same as the GPU test
BLENDER = f"{WORK}/blender/blender"
t_setup = time.time()
if not os.path.exists(BLENDER):
    os.makedirs(f"{WORK}/blender", exist_ok=True)
    subprocess.run(f"wget -q {BLENDER_URL} -O /tmp/b.tar.xz && tar -xf /tmp/b.tar.xz -C {WORK}/blender --strip-components=1 "
                   "&& rm /tmp/b.tar.xz", shell=True, check=True)
subprocess.run([BLENDER, "-b", "--version"], check=True)

# The pack root holds blender/ and blender/out/frozen_FK.blend. The input is read-only, so copy it.
PACK = f"{WORK}/pack/talanta_render_pack"
if not os.path.exists(PACK):
    os.makedirs(f"{WORK}/pack", exist_ok=True)
    unpacked = glob.glob("/kaggle/input/**/talanta_render_pack/blender/out/frozen_FK.blend", recursive=True)
    if unpacked:
        shutil.copytree(unpacked[0].rsplit("/blender/", 1)[0], PACK)
    else:
        parts = sorted(q for q in glob.glob("/kaggle/input/**/talanta_render_pack.tar*", recursive=True) if not q.endswith(".sha256"))
        print(parts)
        subprocess.run(f"cat {' '.join(parts)} | tar -x -C {WORK}/pack", shell=True, check=True)
print(os.listdir(PACK))

In [ ]:
# The inline gpu_pre.py of the GPU test notebook (same as blender/kaggle/gpu_pre.py in the repo).
GPU_PRE = f"{WORK}/gpu_pre.py"
open(GPU_PRE, "w").write('''prefs = bpy.context.preferences.addons["cycles"].preferences
chosen = None
for kind in ("OPTIX", "CUDA"):
    try:
        prefs.compute_device_type = kind
    except TypeError:
        continue
    prefs.get_devices()
    if any(d.type == kind for d in prefs.devices):
        chosen = kind
        break
if chosen is None:
    raise SystemExit("gpu_pre.py: no OptiX or CUDA device found")
for d in prefs.devices:
    d.use = d.type == chosen
s.cycles.device = "GPU"
print("gpu_pre.py:", chosen, [d.name for d in prefs.devices if d.use])
''')
print("setup", round(time.time() - t_setup), "s")

In [ ]:
# One Blender process for the whole range (the first frame of each process takes about 229 s for the OptiX kernels).
# The renderer writes inside the pack; this folder is a link to /kaggle/working, so the frames are the version output.
OUT = f"/kaggle/working/frames/{RUN}"
os.makedirs(OUT, exist_ok=True)
os.makedirs(f"{PACK}/blender/out/kaggle", exist_ok=True)
link = f"{PACK}/blender/out/kaggle/{RUN}"
if not os.path.islink(link):
    os.symlink(OUT, link)
args = [f"blender/out/kaggle/{RUN}", f"chapter={CHAPTER}", f"only={FRAME_START}-{FRAME_END}", f"samples={SAMPLES}", f"haze={HAZE}"]
if RES: args.append(f"res={RES}")
if SHIFT_X is not None: args.append(f"shift_x={SHIFT_X}")
env = dict(os.environ, TALANTA_SITE_BLEND=f"{PACK}/blender/out/frozen_FK.blend", TALANTA_PRE=GPU_PRE, PYTHONUNBUFFERED="1")
cmd = [BLENDER, "-b", "--factory-startup", "--python-exit-code", "1",
       "--python", f"{PACK}/blender/scroll_test/render_scroll_frames.py", "--", *args]
print(" ".join(cmd))
t0 = time.time()
subprocess.run(cmd, cwd=PACK, env=env, check=True)
print("run", RUN, round((time.time() - t0) / 3600, 2), "h")

In [ ]:
# Check the frames and write the run record (it goes with the frames to assets/landing_reel_frames).
import hashlib
want = [f"frame_{i:03d}.png" for i in range(FRAME_START, FRAME_END + 1)]
missing = [f for f in want if not os.path.exists(f"{OUT}/{f}")]
log = json.load(open(f"{OUT}/scroll_log.json"))
sec = log["frame_seconds"]
rec = {"run": RUN, "chapter": CHAPTER, "cut": CUT, "frames": [FRAME_START, FRAME_END], "missing": missing,
       "resolution": log["resolution"], "samples": log["samples"], "haze_volume": log["haze_volume"],
       "first_frame_s": sec[0] if sec else None, "mean_s_after_first": round(sum(sec[1:]) / max(len(sec) - 1, 1), 1),
       "render_h": round(sum(sec) / 3600, 2), "setup_s": log["setup_seconds"], "gpu": subprocess.run(
           ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout.split("\n")[:2],
       "sha256": {f: hashlib.sha256(open(f"{OUT}/{f}", "rb").read()).hexdigest() for f in want if f not in missing},
       "claim": "Internal only. People, vehicles and lighting are illustrative (Visualization)."}
json.dump(rec, open(f"{OUT}/run_record.json", "w"), indent=1)
print(json.dumps({k: v for k, v in rec.items() if k != "sha256"}, indent=1))
assert not missing, f"missing frames: {missing}"